In [1]:
import pandas as pd
import sys
import os
import numpy as np


data_name = '10K_accts'
edge_path = '../flib/AMLsim/outputs/'+data_name+'/tx_log.csv'
node_path = '../flib/AMLsim/outputs/'+data_name+'/node_log.csv'

edge_data = pd.read_csv(edge_path)
node_data = pd.read_csv(node_path)

print('Data is loaded')


Data is loaded


In [2]:
# drop column customer_id and account_type from node_data

bool_dict = {True: 1, False: 0}
for key in bool_dict:
    node_data = node_data.replace(key, bool_dict[key])

bank_dict = {'swedbank': 0, 'handelsbanken': 1, 'others': 2}
for key in bank_dict:
    node_data = node_data.replace(key, bank_dict[key])
    edge_data = edge_data.replace(key, bank_dict[key])

node_data = node_data.drop(['CUSTOMER_ID', 'ACCOUNT_TYPE','COUNTRY'], axis=1)
true_labels = node_data[['ACCOUNT_ID','TRUE_LABEL']]
labels = node_data[['ACCOUNT_ID','IS_SAR']]

labels = labels.rename(columns={'ACCOUNT_ID': 'node_id'})
true_labels = true_labels.rename(columns={'ACCOUNT_ID': 'node_id'})

init_balance_feat = node_data[['ACCOUNT_ID','INIT_BALANCE']]
bank_id_feat = node_data[['ACCOUNT_ID','BANK_ID']]

init_balance_feat = init_balance_feat.rename(columns={'ACCOUNT_ID': 'node_id'})
bank_id_feat = bank_id_feat.rename(columns={'ACCOUNT_ID': 'node_id'})
transactions = edge_data[['step','amount','nameOrig','bankOrig','oldbalanceOrig','newbalanceOrig','nameDest','oldbalanceDest','newbalanceDest']]



In [3]:
edge_data = edge_data[edge_data['type'] == 'TRANSFER']

print(edge_data.head())

edge_sar = edge_data[['isSAR']]
transactions = edge_data[['step','amount','nameOrig','bankOrig','oldbalanceOrig','newbalanceOrig','nameDest','oldbalanceDest','newbalanceDest']]
phone_changes = edge_data[['nameOrig','phoneChangesOrig']]

       step      type   amount  nameOrig bankOrig  daysInBankOrig  \
10000     0  TRANSFER   215.44      4731        2               0   
10001     0  TRANSFER   260.08       713        0               0   
10002     0  TRANSFER   289.34      3416        1               0   
10003     0  TRANSFER  1532.96      3212        1               0   
10004     0  TRANSFER    74.09      5136        2               0   

       phoneChangesOrig  oldbalanceOrig  newbalanceOrig  nameDest bankDest  \
10000                 0        18259.80        18044.35        -1     sink   
10001                 0       123060.74       122800.65        -1     sink   
10002                 0       184604.35       184315.01        -1     sink   
10003                 0       130677.43       129144.46        -1     sink   
10004                 0         7565.08         7490.98        -1     sink   

       daysInBankDest  phoneChangesDest  oldbalanceDest  newbalanceDest  \
10000               0                 0  

In [14]:
transaction_features = transactions.groupby('nameOrig').agg({'amount': ['mean', 'sum'], 'nameDest': 'count'}).reset_index()
transaction_features.columns = ['nameOrig', 'avg_amount_sent', 'total_amount_sent', 'total_transactions_sent']
transaction_features['avg_amount_sent'] = transaction_features['avg_amount_sent'].apply(lambda x: round(x, 2))
transaction_features['total_amount_sent'] = transaction_features['total_amount_sent'].apply(lambda x: round(x, 2))

transaction_features = transaction_features[transaction_features['nameOrig'] != -2]
# Rename column nameOrig to node_id
transaction_features = transaction_features.rename(columns={'nameOrig': 'node_id'})


transaction_features_rec = transactions.groupby('nameDest').agg({'amount': ['mean', 'sum'], 'nameOrig': 'count'}).reset_index()
transaction_features_rec.columns = ['nameDest', 'avg_amount_rec', 'total_amount_rec', 'total_transactions_rec']
transaction_features_rec['avg_amount_rec'] = transaction_features_rec['avg_amount_rec'].apply(lambda x: round(x, 2))
transaction_features_rec['total_amount_rec'] = transaction_features_rec['total_amount_rec'].apply(lambda x: round(x, 2))
transaction_features_rec = transaction_features_rec[transaction_features_rec['nameDest'] != -1]
transaction_features_rec = transaction_features_rec.rename(columns={'nameDest': 'node_id'})

transaction_features = pd.merge(transaction_features, transaction_features_rec, on='node_id', how='outer')

# Total number of phone changes and average number of phone changes per transaction for each node

phone_changes_feat = phone_changes.groupby('nameOrig').agg({'phoneChangesOrig': ['mean', 'sum']}).reset_index()
phone_changes_feat.columns = ['nameOrig', 'avg_phone_changes', 'total_phone_changes']
phone_changes_feat['avg_phone_changes'] = phone_changes_feat['avg_phone_changes'].apply(lambda x: round(x, 2))
phone_changes_feat = phone_changes_feat[phone_changes_feat['nameOrig'] != -2]
phone_changes_feat = phone_changes_feat.rename(columns={'nameOrig': 'node_id'})


# Total number of unique accounts sent to and recieved from
unique_accounts_sent = transactions.groupby('nameOrig').agg({'nameDest': 'nunique'}).reset_index()
unique_accounts_sent = unique_accounts_sent.rename(columns={'nameOrig': 'node_id', 'nameDest': 'unique_accounts_sent'})
unique_accounts_sent = unique_accounts_sent[unique_accounts_sent['node_id'] != -2]

unique_accounts_rec = transactions.groupby('nameDest').agg({'nameOrig': 'nunique'}).reset_index()
unique_accounts_rec = unique_accounts_rec.rename(columns={'nameDest': 'node_id', 'nameOrig': 'unique_accounts_rec'})
unique_accounts_rec = unique_accounts_rec[unique_accounts_rec['node_id'] != -1]


sar_nodes = labels[labels['IS_SAR'] == 1]
# Add a feature of how many transactions recieved by SAR nodes
sar_nodes_rec = transactions[transactions['nameDest'].isin(sar_nodes['node_id'])]
sar_nodes_rec = sar_nodes_rec.groupby('nameDest').agg({'nameOrig': 'count'}).reset_index()
sar_nodes_rec = sar_nodes_rec.rename(columns={'nameDest': 'node_id', 'nameOrig': 'total_transactions_rec_sar'})

sar_nodes_sent = transactions[transactions['nameOrig'].isin(sar_nodes['node_id'])]
sar_nodes_sent = sar_nodes_sent.groupby('nameOrig').agg({'nameDest': 'count'}).reset_index()
sar_nodes_sent = sar_nodes_sent.rename(columns={'nameOrig': 'node_id', 'nameDest': 'total_transactions_sent_sar'})


# Merge all features
features = pd.merge(transaction_features, phone_changes_feat, on='node_id', how='outer')
features = pd.merge(features, init_balance_feat, on='node_id', how='outer')
features = pd.merge(features, bank_id_feat, on='node_id', how='outer')

features = pd.merge(features, unique_accounts_sent, on='node_id', how='outer')
features = pd.merge(features, unique_accounts_rec, on='node_id', how='outer')
features = pd.merge(features, sar_nodes_rec, on='node_id', how='outer')
features = pd.merge(features, sar_nodes_sent, on='node_id', how='outer')

features = features.fillna(0)


In [5]:
print(features.head())  


   node_id  avg_amount_sent  total_amount_sent  total_transactions_sent  \
0        0           859.90          165101.29                      192   
1        1          1555.16          286148.99                      184   
2        2          1512.80          299534.57                      198   
3        3          1830.29          329452.17                      180   
4        4          1756.36          379373.71                      216   

   avg_amount_rec  total_amount_rec  total_transactions_rec  \
0        10534.66         136950.58                      13   
1        37289.58         484764.54                      13   
2        41308.61         537011.93                      13   
3        26775.92         348086.96                      13   
4        38508.55         500611.15                      13   

   avg_phone_changes  total_phone_changes  INIT_BALANCE  BANK_ID  \
0                0.0                    0      83582.10        0   
1                0.0              